## I. Các gói thư viện sử dụng

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

In [ ]:
!pip install python-dotenv huggingface_hub datasets underthesea langdetect -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.5 MB/s eta 0:00:00


In [ ]:
import math
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
from langdetect import detect, detect_langs
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from underthesea import pos_tag, word_tokenize

## II. Thiết lập môi trường làm việc
1. Kết nối google drive
2. Nạp biến môi trường từ .env
3. Kết nối Hugging Face

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [ ]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [ ]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

## III. Tải và trích xuất dữ liệu nhãn 0 từ ICCIES-2025-DetectAI/vietnamese_news_human_ai

In [ ]:
dataset_name = "ICCIES-2025-DetectAI/vietnamese_news_human_ai"

In [ ]:
class DatasetBalancer:
    def __init__(self, dataset_name, split='train', target_size=5000, min_len=256, max_len=512, random_state=42):
        self.dataset_name = dataset_name
        self.split = split
        self.target_size = target_size
        self.min_len = min_len
        self.max_len = max_len
        self.random_state = random_state

    def load_data(self):
        dataset = load_dataset(self.dataset_name, split=self.split)
        return dataset.to_pandas()

    def filter_by_length(self, df):
        df['word_count'] = df['Text'].apply(lambda x: len(str(x).split()))
        df_filtered = df[(df['word_count'] >= self.min_len) & (df['word_count'] <= self.max_len)].copy()
        return df_filtered

    def split_pools(self, df_filtered):
        df_0_pool = df_filtered[df_filtered['Label'] == 0]
        df_1_pool = df_filtered[df_filtered['Label'] == 1]
        return df_0_pool, df_1_pool

    def calculate_target_n(self, df_0_pool, df_1_pool):
        n_0 = min(self.target_size, len(df_0_pool))
        n_1 = min(self.target_size, len(df_1_pool))
        return min(n_0, n_1)

    def sample_stratified(self, df_0_pool, df_1_pool, df_filtered, final_n):
        bins = np.linspace(self.min_len, self.max_len, 11)
        df_filtered['length_bin'] = pd.cut(df_filtered['word_count'], bins=bins, labels=False, include_lowest=True)

        sample_1_df = df_1_pool.sample(n=final_n, random_state=self.random_state)
        bin_counts = sample_1_df['word_count'].apply(lambda x: np.digitize(x, bins)).value_counts(normalize=True)

        sample_0_list = []
        for bin_idx, ratio in bin_counts.items():
            count_to_take = int(round(ratio * final_n))
            bin_val = bins[bin_idx-1] if bin_idx > 0 else bins[0]
            bin_mask = (df_0_pool['word_count'] >= bins[bin_idx-1]) & (df_0_pool['word_count'] < bins[bin_idx])
            df_0_bin = df_0_pool[bin_mask]

            take = min(len(df_0_bin), count_to_take)
            if take > 0:
                sample_0_list.append(df_0_bin.sample(n=take, random_state=self.random_state))

        sample_0_df = pd.concat(sample_0_list)

        if len(sample_0_df) < final_n:
            remaining_needed = final_n - len(sample_0_df)
            extra_samples = df_0_pool.drop(sample_0_df.index).sample(n=remaining_needed, random_state=self.random_state)
            sample_0_df = pd.concat([sample_0_df, extra_samples])
        elif len(sample_0_df) > final_n:
            sample_0_df = sample_0_df.sample(n=final_n, random_state=self.random_state)

        return sample_0_df, sample_1_df

    def finalize_dataset(self, sample_0_df, sample_1_df):
        final_df = pd.concat([sample_0_df, sample_1_df]).sample(frac=1, random_state=self.random_state).reset_index(drop=True)

        if 'length_bin' in final_df.columns:
            final_df = final_df.drop(columns=['length_bin'])

        final_df = final_df.drop(columns=['word_count'])
        return final_df

    def process(self):
        df = self.load_data()
        df_filtered = self.filter_by_length(df)
        df_0_pool, df_1_pool = self.split_pools(df_filtered)
        final_n = self.calculate_target_n(df_0_pool, df_1_pool)
        sample_0_df, sample_1_df = self.sample_stratified(df_0_pool, df_1_pool, df_filtered, final_n)
        final_df = self.finalize_dataset(sample_0_df, sample_1_df)
        return final_df

In [ ]:
balancer = DatasetBalancer(dataset_name=dataset_name)
final_df = balancer.process()

In [ ]:
os.makedirs("Data", exist_ok=True)
save_path = "Data/vietnamese_news_human_ai.csv"
final_df.to_csv(save_path, index=False, encoding='utf-8-sig')

## IV. Phân tích dữ liệu

### 1. Phân tích cơ bản (Có lọc mẫu)

In [ ]:
class TextDataAnalyzer:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = pd.read_csv(self.file_path)

    def add_text_features(self, text_col):
        texts = self.df[text_col].fillna("").astype(str)
        self.df['char_count'] = texts.apply(len)
        self.df['word_count'] = texts.apply(lambda x: len(x.split()))
        self.df['sentence_count'] = texts.apply(
            lambda x: max(1, len([s for s in re.split(r'[.!?]+', x) if s.strip()]))
        )


    def filter_by_word_count(self, text_col, min_words, max_words):
        if 'word_count' in self.df.columns:
            word_counts = self.df['word_count']
        else:
            word_counts = self.df[text_col].astype(str).apply(lambda x: len(x.split()))

        self.df = self.df[(word_counts >= min_words) & (word_counts <= max_words)]
        self.df = self.df.reset_index(drop=True)

        return len(self.df)

    def get_basic_stats(self, text_col, label_col="Label"):
        missing_counts = self.df.isnull().sum()

        label_counts = {}
        if label_col in self.df.columns:
            label_counts = self.df[label_col].value_counts().to_dict()

        return {
            "total_rows": len(self.df),
            "columns": list(self.df.columns),
            "duplicates": self.df.duplicated().sum(),
            "total_missing": missing_counts.sum(),
            "missing_details": missing_counts[missing_counts > 0].to_dict(),
            "empty_strings": (self.df[text_col].astype(str).str.strip() == "").sum(),
            "label_counts": label_counts
        }

    def get_text_stats(self, text_col):
        if self.df.empty:
            return {"min_len": 0, "max_len": 0, "mean_len": 0, "outliers_count": 0}

        if 'word_count' in self.df.columns:
            word_counts = self.df['word_count']
        else:
            word_counts = self.df[text_col].astype(str).apply(lambda x: len(x.split()))

        q1 = word_counts.quantile(0.25)
        q3 = word_counts.quantile(0.75)
        iqr = q3 - q1

        return {
            "min_len": word_counts.min(),
            "max_len": word_counts.max(),
            "mean_len": word_counts.mean(),
            "outliers_count": ((word_counts < q1 - 1.5 * iqr) | (word_counts > q3 + 1.5 * iqr)).sum()
        }

class ReportViewer:
    @staticmethod
    def display_report(file_path, text_col, basic_stats, text_stats):
        print(f"File: {file_path}")
        print("-" * 50)

        print("BÁO CÁO THỐNG KÊ CƠ BẢN")
        print(f"1. Tổng số mẫu (dòng): {basic_stats['total_rows']:,}")
        print(f"2. Các thuộc tính (cột): {basic_stats['columns']}")
        print(f"3. Dữ liệu trùng lặp: {basic_stats['duplicates']:,} dòng")
        print(f"4. Tổng giá trị Null/NaN: {basic_stats['total_missing']:,}")

        for col, count in basic_stats['missing_details'].items():
            print(f"   -> Cột '{col}' có {count:,} giá trị thiếu")

        print(f"5. Dữ liệu rỗng ở cột '{text_col}': {basic_stats['empty_strings']:,} dòng")

        if basic_stats.get('label_counts'):
            print("6. Phân bổ nhãn (Label):")
            for label, count in sorted(basic_stats['label_counts'].items()):
                print(f"   -> Label {label}: {count:,} mẫu")

        print("-" * 50)
        print("THỐNG KÊ CHIỀU DÀI VĂN BẢN")
        print(f"- Ngắn nhất: {text_stats['min_len']:,} từ")
        print(f"- Dài nhất: {text_stats['max_len']:,} từ")
        print(f"- Trung bình: {text_stats['mean_len']:,.2f} từ")
        print(f"- Số lượng Outliers (theo IQR): {text_stats['outliers_count']:,} mẫu")

In [ ]:
analyzer = TextDataAnalyzer("Data/vietnamese_news_human_ai.csv")
data_column = "Text"
analyzer.add_text_features(text_col=data_column)

rows_after_filter = analyzer.filter_by_word_count(text_col=data_column, min_words=256, max_words=512)
basic_info = analyzer.get_basic_stats(text_col=data_column)
text_info = analyzer.get_text_stats(text_col=data_column)

ReportViewer.display_report(analyzer.file_path, data_column, basic_info, text_info)

File: Data/vietnamese_news_human_ai.csv
--------------------------------------------------
BÁO CÁO THỐNG KÊ CƠ BẢN
1. Tổng số mẫu (dòng): 10,000
2. Các thuộc tính (cột): ['Text', 'Label', 'char_count', 'word_count', 'sentence_count']
3. Dữ liệu trùng lặp: 0 dòng
4. Tổng giá trị Null/NaN: 0
5. Dữ liệu rỗng ở cột 'Text': 0 dòng
6. Phân bổ nhãn (Label):
   -> Label 0: 5,000 mẫu
   -> Label 1: 5,000 mẫu
--------------------------------------------------
THỐNG KÊ CHIỀU DÀI VĂN BẢN
- Ngắn nhất: 258 từ
- Dài nhất: 511 từ
- Trung bình: 364.53 từ
- Số lượng Outliers (theo IQR): 0 mẫu


### 2. Phân tích nâng cao (đặc trưng)

In [ ]:
DEFAULT_N_GRAM = 5
DEFAULT_TOP_K = 10

In [ ]:
class TextFeatureExtractor:
    def __init__(self, df, text_col):
        self.df = df.dropna(subset=[text_col]).copy()
        self.text_col = text_col
        self.texts = self.df[text_col].astype(str)

    def compute_lexical_richness(self): # Type-Token Ratio - TTR
        def calc_ttr(text):
            tokens = text.lower().split()
            if not tokens:
                return 0.0
            types = set(tokens)
            return len(types) / len(tokens)

        self.df['ttr'] = self.texts.apply(calc_ttr)

        return self.df['ttr'].mean()

    def compute_structural_features(self):
        self.df['sentence_count'] = self.texts.apply(lambda x: max(1, len(re.split(r'[.!?]+', x)) - 1))
        self.df['word_count'] = self.texts.apply(lambda x: len(x.split()))
        self.df['number_count'] = self.texts.apply(lambda x: len(re.findall(r'\d+', x)))
        self.df['capitalized_words'] = self.texts.apply(lambda x: len(re.findall(r'\b[A-Z][a-z]*\b', x)))

        return {
            "avg_sentences": self.df['sentence_count'].mean(),
            "avg_words_per_sentence": (self.df['word_count'] / self.df['sentence_count']).mean(),
            "avg_numbers_per_text": self.df['number_count'].mean(),
            "avg_capitalized_words": self.df['capitalized_words'].mean()
        }

    def get_top_ngrams(self, n=DEFAULT_N_GRAM, top_k=DEFAULT_TOP_K): # N-gram phổ biến
        ngram_counts = Counter()
        for text in self.texts:
            tokens = re.findall(r'\b[a-z_àáãạảăắằẳẵặâấầẩẫậèéẹẻẽêềếểễệđìíĩỉịòóõọỏôốồổỗộơớờởỡợùúũụủưứừửữựỳỵỷỹ]+\b', text.lower())
            ngrams = zip(*[tokens[i:] for i in range(n)])

            for ngram in ngrams: # Nối cụm N-gram
                ngram_counts[" ".join(ngram)] += 1

        return ngram_counts.most_common(top_k)

class AdvancedReportViewer:
    @staticmethod
    def display_advanced_report(lexical_richness, structural_features, bigrams, trigrams):
        print("BÁO CÁO PHÂN TÍCH ĐẶC TRƯNG")
        print("-" * 65)

        print("1. ĐỘ PHONG PHÚ VÀ CẤU TRÚC NGÔN NGỮ:")
        print(f"   - Chỉ số đa dạng từ vựng (TTR trung bình): {lexical_richness:.4f}")
        print(f"   - Trung bình số câu / bài viết: {structural_features['avg_sentences']:.1f} câu")
        print(f"   - Trung bình số từ / câu: {structural_features['avg_words_per_sentence']:.1f} từ")
        print(f"   - Trung bình số lượng con số / bài: {structural_features['avg_numbers_per_text']:.1f} số")
        print(f"   - Trung bình số từ viết hoa / bài: {structural_features['avg_capitalized_words']:.1f} từ")

        print("\n2. PHÂN TÍCH XU HƯỚNG CỤM TỪ (N-GRAMS):")
        print("   -> Top 10 Cụm 2 từ xuất hiện nhiều nhất:")
        for phrase, count in bigrams:
            print(f"      + '{phrase}': {count:,} lần")

        print("   -> Top 10 Cụm 3 từ (Tri-grams) xuất hiện nhiều nhất:")
        for phrase, count in trigrams:
            print(f"      + '{phrase}': {count:,} lần")

In [ ]:
extractor = TextFeatureExtractor(df=analyzer.df, text_col="Text")

ttr = extractor.compute_lexical_richness()
structural = extractor.compute_structural_features()
top_bigrams = extractor.get_top_ngrams(n=2, top_k=10)
top_trigrams = extractor.get_top_ngrams(n=3, top_k=10)

AdvancedReportViewer.display_advanced_report(
    lexical_richness=ttr,
    structural_features=structural,
    bigrams=top_bigrams,
    trigrams=top_trigrams,
)

analyzer.df = extractor.df

BÁO CÁO PHÂN TÍCH ĐẶC TRƯNG
-----------------------------------------------------------------
1. ĐỘ PHONG PHÚ VÀ CẤU TRÚC NGÔN NGỮ:
   - Chỉ số đa dạng từ vựng (TTR trung bình): 0.6116
   - Trung bình số câu / bài viết: 12.6 câu
   - Trung bình số từ / câu: 32.4 từ
   - Trung bình số lượng con số / bài: 7.3 số
   - Trung bình số từ viết hoa / bài: 14.5 từ

2. PHÂN TÍCH XU HƯỚNG CỤM TỪ (N-GRAMS):
   -> Top 10 Cụm 2 từ xuất hiện nhiều nhất:
      + 'việt nam': 6,975 lần
      + 'là một': 6,477 lần
      + 'không chỉ': 6,365 lần
      + 'có thể': 6,188 lần
      + 'mà còn': 5,341 lần
      + 'phát triển': 5,050 lần
      + 'thực hiện': 4,617 lần
      + 'đặc biệt': 4,523 lần
      + 'quan trọng': 4,083 lần
      + 'chủ tịch': 3,739 lần
   -> Top 10 Cụm 3 từ (Tri-grams) xuất hiện nhiều nhất:
      + 'mà còn là': 1,895 lần
      + 'này không chỉ': 1,787 lần
      + 'không chỉ là': 1,763 lần
      + 'đặc biệt là': 1,627 lần
      + 'một trong những': 1,449 lần
      + 'chủ tịch nước': 1,211 

### 3. Phân tích độ nhiễu

In [ ]:
class TextNoiseAnalyzer:
    def __init__(self, df, text_col):
        self.df = df.dropna(subset=[text_col]).copy()
        self.text_col = text_col
        self.texts = self.df[text_col].astype(str)

    def analyze_formatting_noise(self):
        self.df['email_count'] = self.texts.apply(lambda x: len(re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', x)))
        return {"avg_emails": self.df['email_count'].mean()}

    def analyze_lexical_noise(self):
        self.df['long_words_count'] = self.texts.apply(lambda x: len(re.findall(r'\b\w{15,}\b', x)))
        vietnamese_chars_pattern = r'[^a-zA-Z0-9\s_àáãạảăắằẳẵặâấầẩẫậèéẹẻẽêềếểễệđìíĩỉịòóõọỏôốồổỗộơớờởỡợùúũụủưứừửữựỳỵỷỹÀÁÃẠẢĂẮẰẲẴẶÂẤẦẨẪẬÈÉẸẺẼÊỀẾỂỄỆĐÌÍĨỈỊÒÓÕỌỎÔỐỒỔỖỘƠỚỜỞỠỢÙÚŨỤỦƯỨỪỬỮỰỲỴỶỸ]'
        special_chars = self.texts.apply(lambda x: len(re.findall(vietnamese_chars_pattern, x)))
        text_lengths = self.texts.apply(len).replace(0, 1)
        self.df['special_char_ratio'] = special_chars / text_lengths

        return {
            "avg_long_words": self.df['long_words_count'].mean(),
            "special_char_ratio": self.df['special_char_ratio'].mean()
        }

    def analyze_syntactic_noise(self):
        self.df['caps_ratio'] = self.texts.apply(lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0)
        self.df['missing_punctuation'] = self.texts.apply(lambda x: 1 if len(re.findall(r'[.!?]', x)) == 0 else 0)

        return {
            "avg_caps_ratio": self.df['caps_ratio'].mean(),
            "missing_punctuation_ratio": self.df['missing_punctuation'].mean()
        }

class NoiseReportViewer:
    @staticmethod
    def display_noise_report(formatting, lexical, syntactic):
        print("BÁO CÁO PHÂN TÍCH ĐỘ NHIỄU DỮ LIỆU")
        print("-" * 65)

        print("1. NHIỄU ĐỊNH DẠNG:")
        print(f"   - Trung bình Email / bài: {formatting['avg_emails']:.2f} email")

        print("\n2. NHIỄU TỪ VỰNG & KÝ TỰ:")
        print(f"   - Trung bình từ dính chùm (> 15 ký tự) / bài: {lexical['avg_long_words']:.2f} từ")
        print(f"   - Tỷ lệ ký tự lạ / đặc biệt: {lexical['special_char_ratio']:.4%}")

        print("\n3. NHIỄU CÚ PHÁP:")
        print(f"   - Tỷ lệ chữ IN HOA toàn văn bản: {syntactic['avg_caps_ratio']:.4%}")
        print(f"   - Tỷ lệ bài viết KHÔNG CÓ dấu chấm câu: {syntactic['missing_punctuation_ratio']:.4%}")

In [ ]:
noise_analyzer = TextNoiseAnalyzer(df=analyzer.df, text_col="Text")

formatting_noise = noise_analyzer.analyze_formatting_noise()
lexical_noise = noise_analyzer.analyze_lexical_noise()
syntactic_noise = noise_analyzer.analyze_syntactic_noise()

NoiseReportViewer.display_noise_report(formatting_noise, lexical_noise, syntactic_noise)
analyzer.df = noise_analyzer.df

BÁO CÁO PHÂN TÍCH ĐỘ NHIỄU DỮ LIỆU
-----------------------------------------------------------------
1. NHIỄU ĐỊNH DẠNG:
   - Trung bình Email / bài: 0.01 email

2. NHIỄU TỪ VỰNG & KÝ TỰ:
   - Trung bình từ dính chùm (> 15 ký tự) / bài: 0.02 từ
   - Tỷ lệ ký tự lạ / đặc biệt: 2.6868%

3. NHIỄU CÚ PHÁP:
   - Tỷ lệ chữ IN HOA toàn văn bản: 2.9885%
   - Tỷ lệ bài viết KHÔNG CÓ dấu chấm câu: 0.0500%


### 4. Xét đặc trưng quan trọng

In [ ]:
class VietnameseDataProcessor:
    def __init__(self, custom_stopwords=None):
        self.stopwords = set()
        self.load_stopwords(custom_stopwords)


    def load_stopwords(self, file_path):
        if os.path.exists(file_path):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    self.stopwords = set(line.strip().replace(' ', '_') for line in f if line.strip())
            except Exception as e:
                self.stopwords = set(["và", "thì", "là", "mà", "của", "đã", "đang", "sẽ", "để", "các", "những", "một"])
        else:
            self.stopwords = set(["và", "thì", "là", "mà", "của", "đã", "đang", "sẽ", "để", "các", "những", "một"])

    def segment_words(self, text):
        if not text or not isinstance(text, str):
            return ""

        return word_tokenize(text, format="text")

    def calculate_stopwords_ratio(self, text):
        tokens = text.split()
        if not tokens:
            return 0.0, []

        stopword_tokens = [word for word in tokens if word.lower() in self.stopwords]
        ratio = (len(stopword_tokens) / len(tokens)) * 100

        return ratio, stopword_tokens

    def detect_language(self, text):
        try:
            main_lang = detect(text)
            return main_lang
        except:
            return "unknown"

    def analyze_document(self, text):
        segmented_text = self.segment_words(text)
        sw_ratio, sw_found = self.calculate_stopwords_ratio(segmented_text)
        lang = self.detect_language(text)

        return {
            "original_text": text,
            "segmented_text": segmented_text,
            "stopwords_ratio_percent": round(sw_ratio, 2),
            "stopwords_found": sw_found,
            "main_language": lang,
            "is_valid_vietnamese": lang == "vi"
        }

In [ ]:
class DatasetNLPAnalyzer:
    def __init__(self, dataframe, text_col, nlp_processor):
        self.df = dataframe
        self.text_col = text_col
        self.processor = nlp_processor
        self.stats = {}

    def process_data(self):
        self.df[self.text_col] = self.df[self.text_col].fillna("")

        self.df['nlp_results'] = self.df[self.text_col].apply(
            lambda x: self.processor.analyze_document(str(x))
        )

        self.df['segmented_text'] = self.df['nlp_results'].apply(lambda x: x['segmented_text'])
        self.df['stopwords_ratio'] = self.df['nlp_results'].apply(lambda x: x['stopwords_ratio_percent'])
        self.df['language'] = self.df['nlp_results'].apply(lambda x: x['main_language'])
        self.df['is_valid_vi'] = self.df['nlp_results'].apply(lambda x: x['is_valid_vietnamese'])
        self.df['stopwords_found'] = self.df['nlp_results'].apply(lambda x: x['stopwords_found'])

        self.df = self.df.drop(columns=['nlp_results'])

    def calculate_statistics(self):
        self.stats['total_docs'] = len(self.df)
        self.stats['lang_stats'] = self.df['language'].value_counts(normalize=True) * 100
        self.stats['valid_vi_percent'] = self.df['is_valid_vi'].mean() * 100
        self.stats['avg_stopwords_ratio'] = self.df['stopwords_ratio'].mean()

        all_stopwords = []
        for sw_list in self.df['stopwords_found']:
            all_stopwords.extend(sw_list)
        self.stats['top_10_stopwords'] = Counter(all_stopwords).most_common(10)

    def print_report(self):
        print("\n" + "-"*50)
        print(" BÁO CÁO THỐNG KÊ XỬ LÝ NGÔN NGỮ TỰ NHIÊN (NLP)")
        print("-"*50)
        print(f"Tổng số văn bản đã xử lý: {self.stats['total_docs']:,} dòng")

        print("\n1. THỐNG KÊ TÌNH TRẠNG NGÔN NGỮ:")
        print(f" - Tỷ lệ văn bản Tiếng Việt (Hợp lệ): {self.stats['valid_vi_percent']:.2f}%")
        print(f" - Tỷ lệ ngôn ngữ ngoại lai / nhiễu: {(100 - self.stats['valid_vi_percent']):.2f}%")
        print(" - Chi tiết phân bổ các mã ngôn ngữ:")
        for lang, pct in self.stats['lang_stats'].items():
            print(f"   + Mã '{lang}': {pct:.2f}%")

        print("\n2. THỐNG KÊ TỶ LỆ STOPWORDS (HƯ TỪ):")
        print(f" - Tỷ lệ Stopwords trung bình trên toàn tập: {self.stats['avg_stopwords_ratio']:.2f}%")
        print(" - Top 10 Stopwords làm nhiễu dữ liệu nhiều nhất:")
        for word, count in self.stats['top_10_stopwords']:
            print(f"   + Từ '{word}': {count:,} lần xuất hiện")

    def run_pipeline(self):
        self.process_data()
        self.calculate_statistics()
        self.print_report()

        return self.df

In [ ]:
processor = VietnameseDataProcessor(custom_stopwords="Data/vietnamese-stopwords.txt")

dataset_analyzer = DatasetNLPAnalyzer(
    dataframe=analyzer.df,
    text_col='Text',
    nlp_processor=processor
)

analyzer.df = dataset_analyzer.run_pipeline()


--------------------------------------------------
 BÁO CÁO THỐNG KÊ XỬ LÝ NGÔN NGỮ TỰ NHIÊN (NLP)
--------------------------------------------------
Tổng số văn bản đã xử lý: 10,000 dòng

1. THỐNG KÊ TÌNH TRẠNG NGÔN NGỮ:
 - Tỷ lệ văn bản Tiếng Việt (Hợp lệ): 100.00%
 - Tỷ lệ ngôn ngữ ngoại lai / nhiễu: 0.00%
 - Chi tiết phân bổ các mã ngôn ngữ:
   + Mã 'vi': 100.00%

2. THỐNG KÊ TỶ LỆ STOPWORDS (HƯ TỪ):
 - Tỷ lệ Stopwords trung bình trên toàn tập: 41.97%
 - Top 10 Stopwords làm nhiễu dữ liệu nhiều nhất:
   + Từ 'và': 58,058 lần xuất hiện
   + Từ 'của': 48,726 lần xuất hiện
   + Từ 'là': 32,304 lần xuất hiện
   + Từ 'những': 30,411 lần xuất hiện
   + Từ 'trong': 30,111 lần xuất hiện
   + Từ 'một': 28,818 lần xuất hiện
   + Từ 'các': 26,795 lần xuất hiện
   + Từ 'đã': 25,091 lần xuất hiện
   + Từ 'cho': 24,891 lần xuất hiện
   + Từ 'được': 23,529 lần xuất hiện


In [ ]:
clean_df = analyzer.df[analyzer.df['is_valid_vi'] == True]
pd.set_option('display.max_colwidth', None)
clean_df.head(2)


,Text,Label,char_count,word_count,sentence_count,ttr,number_count,capitalized_words,email_count,long_words_count,special_char_ratio,caps_ratio,missing_punctuation,segmented_text,stopwords_ratio,language,is_valid_vi,stopwords_found
0,"Trầm cảm là một căn bệnh tâm lý phổ biến, có thể được phân loại thành nhiều mức độ khác nhau như nhẹ, trung bình và nặng. Khi người bệnh đến bệnh viện để thăm khám, họ sẽ được thực hiện một bài kiểm tra dựa trên bảng hỏi PHQ-9 nhằm xác định mức độ nghiêm trọng của tình trạng bệnh. Một yếu tố quan trọng để quá trình điều trị đạt hiệu quả cao là người bệnh cần nhận thức rõ ràng về việc họ cần được can thiệp y tế. \n\nViệc điều trị trầm cảm thường bao gồm việc sử dụng thuốc, đặc biệt là các loại thuốc an thần và thuốc chống trầm cảm. Tuy nhiên, chỉ những bác sĩ có giấy phép hành nghề chuyên khoa tâm thần mới có quyền kê đơn thuốc, và họ cũng phải đăng ký chữ ký với sở y tế. Theo bác sĩ CKII Trần Minh Khuyên, thuốc chống trầm cảm thường phát huy tác dụng tốt nhất sau khoảng 2 đến 3 tuần sử dụng. Khi các triệu chứng giảm từ 80 đến 90%, bác sĩ sẽ xem xét giảm liều lượng thuốc và yêu cầu bệnh nhân tái khám định kỳ. Việc ngừng thuốc đột ngột hoặc tự ý mua thuốc bên ngoài có thể dẫn đến những tác dụng phụ nghiêm trọng và không giúp bệnh nhân khỏi bệnh.\n\nVề các dịch vụ tư vấn trầm cảm trực tuyến, bác sĩ Khuyên nhấn mạnh rằng những dịch vụ này không thể thay thế cho việc điều trị y tế. Những người thực hiện tư vấn thường chỉ là các chuyên gia tâm lý, không phải bác sĩ. Tại Việt Nam, không có chức danh ""bác sĩ tâm lý"", và các trung tâm tư vấn thường sử dụng đội ngũ cử nhân hoặc thạc sĩ tâm lý để cung cấp dịch vụ tư vấn, nhưng họ chỉ dừng lại ở mức độ tư vấn về tâm lý giáo dục và xã hội. Việc điều trị trầm cảm cần phải được thực hiện bởi các bác sĩ có chuyên môn và giấy phép hợp pháp.",1,1596,356,12,0.570225,5,6,0,0,0.023183,0.015664,0,"Trầm_cảm là một căn_bệnh_tâm_lý phổ_biến , có_thể được phân_loại thành nhiều mức_độ khác nhau như nhẹ , trung_bình và nặng . Khi người_bệnh đến bệnh_viện để thăm_khám , họ sẽ được thực_hiện một bài kiểm_tra dựa trên bảng hỏi PHQ-9 nhằm xác_định mức_độ nghiêm_trọng của tình_trạng bệnh . Một yếu_tố quan_trọng để quá_trình điều_trị đạt hiệu_quả cao là người_bệnh cần nhận_thức rõ_ràng về việc họ cần được can_thiệp y_tế . Việc điều_trị trầm_cảm thường bao_gồm việc sử_dụng thuốc , đặc_biệt là các loại thuốc an_thần và thuốc chống trầm_cảm . Tuy_nhiên , chỉ những bác_sĩ có giấy_phép hành_nghề chuyên_khoa tâm_thần mới có quyền kê đơn thuốc , và họ cũng phải đăng_ký chữ_ký với sở y_tế . Theo bác_sĩ CKII Trần_Minh_Khuyên , thuốc chống trầm_cảm thường phát_huy tác_dụng tốt nhất sau khoảng 2 đến 3 tuần sử_dụng . Khi các triệu_chứng giảm từ 80 đến 90 % , bác_sĩ sẽ xem_xét giảm liều_lượng thuốc và yêu_cầu bệnh_nhân tái khám định_kỳ . Việc ngừng thuốc đột_ngột hoặc tự_ý mua thuốc bên ngoài có_thể dẫn đến những tác_dụng phụ nghiêm_trọng và không giúp bệnh_nhân khỏi bệnh . Về các dịch_vụ tư_vấn trầm_cảm trực_tuyến , bác_sĩ Khuyên nhấn_mạnh rằng những dịch_vụ này không_thể thay_thế cho việc điều_trị y_tế . Những người thực_hiện tư_vấn thường chỉ là các chuyên_gia tâm_lý , không phải bác_sĩ . Tại Việt_Nam , không có chức_danh "" bác_sĩ tâm_lý "" , và các trung_tâm tư_vấn thường sử_dụng đội_ngũ cử_nhân hoặc thạc_sĩ tâm_lý để cung_cấp dịch_vụ tư_vấn , nhưng họ chỉ dừng lại ở mức_độ tư_vấn về tâm_lý giáo_dục và xã_hội . Việc điều_trị trầm_cảm cần phải được thực_hiện bởi các bác_sĩ có chuyên_môn và giấy_phép hợp_pháp .",46.04,vi,True,"[là, một, có_thể, được, nhiều, khác, nhau, như, và, nặng, Khi, đến, để, họ, sẽ, được, thực_hiện, một, bài, trên, hỏi, nhằm, của, tình_trạng, Một, quan_trọng, để, quá_trình, đạt, cao, là, cần, về, việc, họ, cần, được, Việc, thường, việc, sử_dụng, đặc_biệt, là, các, loại, và, Tuy_nhiên, chỉ, những, có, mới, có, và, họ, cũng, phải, với, Theo, thường, tốt, nhất, sau, khoảng, đến, sử_dụng, Khi, các, giảm, từ, đến, sẽ, giảm, và, yêu_cầu, Việc, hoặc, tự_ý, bên, ngoài, có_t

In [ ]:
pd.reset_option('display.max_colwidth')

In [ ]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Text                 10000 non-null  object 
 1   Label                10000 non-null  int64  
 2   char_count           10000 non-null  int64  
 3   word_count           10000 non-null  int64  
 4   sentence_count       10000 non-null  int64  
 5   ttr                  10000 non-null  float64
 6   number_count         10000 non-null  int64  
 7   capitalized_words    10000 non-null  int64  
 8   email_count          10000 non-null  int64  
 9   long_words_count     10000 non-null  int64  
 10  special_char_ratio   10000 non-null  float64
 11  caps_ratio           10000 non-null  float64
 12  missing_punctuation  10000 non-null  int64  
 13  segmented_text       10000 non-null  object 
 14  stopwords_ratio      10000 non-null  float64
 15  language             10000 non-null  

### V. Bổ sung đặc trưng Perplexity, Entropy, POS, Burstiness  

In [ ]:
class AdvancedLinguisticExtractor:
    def __init__(self, df, text_col, model_name="NlpHUST/gpt2-vietnamese"):
        from transformers import logging
        logging.set_verbosity_error()

        self.df = df.copy()
        self.text_col = text_col
        self.texts = self.df[text_col].astype(str)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        ).to(self.device)
        self.model.eval()

    @torch.no_grad()
    def compute_entropy_and_perplexity(self, text):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return 0.0, 0.0

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
            padding=False
        )

        input_ids = inputs["input_ids"].to(self.device)

        if input_ids.size(1) < 2:
            return 0.0, 0.0

        outputs = self.model(input_ids, labels=input_ids)
        loss = outputs.loss.item()

        entropy = loss / math.log(2)
        perplexity = math.exp(loss)

        return entropy, perplexity

    def compute_pos_distribution(self, text):
        if not text.strip():
            return {}

        tags = pos_tag(text)
        total_tags = len(tags)

        if total_tags == 0:
            return {}

        tag_counts = Counter([tag for word, tag in tags])

        return {
            'noun_ratio': tag_counts.get('N', 0) / total_tags,
            'verb_ratio': tag_counts.get('V', 0) / total_tags,
            'adj_ratio': tag_counts.get('A', 0) / total_tags,
            'adverb_ratio': tag_counts.get('R', 0) / total_tags,
            'pronoun_ratio': tag_counts.get('P', 0) / total_tags,
            'conjunction_ratio': tag_counts.get('C', 0) / total_tags
        }

    def extract_features(self):
        tqdm.pandas(desc="Đang xử lý")

        ep_results = self.texts.progress_apply(self.compute_entropy_and_perplexity)

        self.df['entropy'] = [res[0] for res in ep_results]
        self.df['perplexity'] = [res[1] for res in ep_results]

        pos_results = self.texts.progress_apply(self.compute_pos_distribution)
        pos_df = pd.DataFrame(pos_results.tolist()).fillna(0)
        self.df = pd.concat([self.df, pos_df], axis=1)

        return self.df

In [ ]:
adv_extractor = AdvancedLinguisticExtractor(df=clean_df, text_col="Text")
extracted_df = adv_extractor.extract_features()
final_clean_df = extracted_df
final_clean_df.info()

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: NlpHUST/gpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]


Đang xử lý:   0%|          | 0/10000 [00:00<?, ?it/s]`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.

Đang xử lý: 100%|██████████| 10000/10000 [29:03<00:00,  5.74it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Text                 10000 non-null  object 
 1   Label                10000 non-null  int64  
 2   char_count           10000 non-null  int64  
 3   word_count           10000 non-null  int64  
 4   sentence_count       10000 non-null  int64  
 5   ttr                  10000 non-null  float64
 6   number_count         10000 non-null  int64  
 7   capitalized_words    10000 non-null  int64  
 8   email_count          10000 non-null  int64  
 9   long_words_count     10000 non-null  int64  
 10  special_char_ratio   10000 non-null  float64
 11  caps_ratio           10000 non-null  float64
 12  missing_punctuation  10000 non-null  int64  
 13  segmented_text       10000 non-null  object 
 14  stopwords_ratio      10000 non-null  float64
 15  language             10000 non-null  

In [ ]:
def calculate_burstiness(text):
    sentences = re.split(r'[.!?]+', str(text))
    sentence_lengths = [len(s.split()) for s in sentences if s.strip()]

    if len(sentence_lengths) <= 1:
        return 0.0

    return np.std(sentence_lengths)

In [ ]:
final_clean_df['burstiness'] = final_clean_df['Text'].apply(calculate_burstiness)

## VI. Lưu thông tin EDA Dataset

In [ ]:
final_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Text                 10000 non-null  object 
 1   Label                10000 non-null  int64  
 2   char_count           10000 non-null  int64  
 3   word_count           10000 non-null  int64  
 4   sentence_count       10000 non-null  int64  
 5   ttr                  10000 non-null  float64
 6   number_count         10000 non-null  int64  
 7   capitalized_words    10000 non-null  int64  
 8   email_count          10000 non-null  int64  
 9   long_words_count     10000 non-null  int64  
 10  special_char_ratio   10000 non-null  float64
 11  caps_ratio           10000 non-null  float64
 12  missing_punctuation  10000 non-null  int64  
 13  segmented_text       10000 non-null  object 
 14  stopwords_ratio      10000 non-null  float64
 15  language             10000 non-null  

In [ ]:
save_path = "Data/vietnamese_news_human_ai_eda.csv"
final_clean_df.to_csv(save_path, index=False, encoding='utf-8-sig')